In [47]:
# Import modules
import selenium # for navigating borgerforslag.dk
import requests # for scraping
import pandas as pd # for data wrapping, cleaning and handling
from bs4 import BeautifulSoup # for parsing
import time # for scraping
import tqdm # for scraping
import random # for scraping
import pprint # for displaying json code in an ordered manner
import re # for recognising patterns in extracted html code
from pathlib import Path # class for defining file path
import csv # for exporting in .csv
import json # for exporting in .json

# Import classes and statements
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By # to use CSS selector for e.g. cookie-clicking
from selenium.webdriver.support.ui import WebDriverWait # for waiting
from selenium.webdriver.support import expected_conditions as EC 
from selenium.common.exceptions import TimeoutException
from selenium.webdriver.common.keys import Keys # for using keyboard keys like RETURN
from selenium.common.exceptions import NoSuchElementException



In [4]:
# directing a Selenium Chrome driver on Borgerforslag.dk

### to use the Selenium Chrome driver
chrome_options = Options()
chrome_options.add_argument("--disable-search-engine-choice-screen")
driver = webdriver.Chrome(options=chrome_options)

### getting on website
url_Borgerforslag = "https://borgerforslag.dk/" # URL til borgerforslag.dk
driver.get(url_Borgerforslag) # we use .get() to open url in Selenium Chrome

### getting through cookies
try:
    cookie = WebDriverWait(driver, 10).until(
        EC.presence_of_element_located((By.ID, 'CybotCookiebotDialogBodyLevelButtonLevelOptinAllowallSelection'))
    )
    cookie.click()
except TimeoutException:
    print("Element not found within the specified wait time.")


### choosing "alle" instead of "igangværende" borgerforslag
try:
    click_1 = WebDriverWait(driver, 5).until(
        EC.presence_of_element_located((By.ID, 'react-select-filter1--value-item'))
    )
    click_1.click()
    click_2 = WebDriverWait(driver, 3).until(
        EC.presence_of_element_located((By.ID, 'react-select-filter1--option-0'))
    )
    click_2.click()
except TimeoutException:
    print("Element not found within the specified wait time.")



In [10]:
### FINAL: going through more 'borgerforslag'

alarm = False  # Defining a stop_alarm

while alarm == False:
    try:
        click_3 = driver.find_element(By.CSS_SELECTOR, 'button[class="dFsu8t fYY1lZ vFact_DoNotReadAloud _3-IJkM _3CrCss"]')
        click_3.click()
        time.sleep(random.uniform(1,5))
    except NoSuchElementException:
        alarm = True
        print("No more pages to go through")


No more pages to go through


In [11]:
soup = BeautifulSoup(driver.page_source,'lxml') # 
all_sites = soup.find_all('a', class_='lQq327') # 
#company = stored_company[3].text # storing the company name by selecting the forth row, in text by using the .text operation

pprint.pprint(all_sites)

[<a class="lQq327" href="https://borgerforslag.dk/se-og-stoet-forslag/?Id=FT-18099" style="opacity: 1; transform: translateY(0px);"><div class="_2kk3hN"><div><span>Antal støtter</span><strong class="_3at1A7">70</strong></div></div><div class="ssLhPH"><div class="_2KXwjO" style="width: 1.87083%;"></div></div><h3 class="Cfb3RR">Fartbøder forhindrer i at opnå Dansk statsborgerskab</h3><div class="_2-6RI9"><span>06. august 2024</span><span>FT-18099</span></div></a>,
 <a class="lQq327" href="https://borgerforslag.dk/se-og-stoet-forslag/?Id=FT-18075" style="opacity: 1; transform: translateY(0px);"><div class="_2kk3hN"><div><span>Antal støtter</span><strong class="_3at1A7">219</strong></div></div><div class="ssLhPH"><div class="_2KXwjO" style="width: 3.30908%;"></div></div><h3 class="Cfb3RR">Forbyd dressurridning som konkurrencesport</h3><div class="_2-6RI9"><span>06. august 2024</span><span>FT-18075</span></div></a>,
 <a class="lQq327" href="https://borgerforslag.dk/se-og-stoet-forslag/?Id=F

In [16]:
### cleaning for URL data, borgerforslag

url_list = [] # creating empty list for urls

for strings in all_sites: # using for-loop to utilize regular expressions on list, all_sites is a list
    converted_strings = str(strings) # convert each element in list to string
    extracted_url = re.findall(r'href="(https?://[^"]+)"', converted_strings) # using regular expressions to extract url pattern
    url_list.append(extracted_url) # append to empty list for urls

# Flattening the list
url_list = [url for sublist in url_list for url in sublist] # very bad solution for flatterning the list


In [17]:
### cleaning for titles data, borgerforslag

titles_list = [] # creating empty list for urls

for strings in all_sites: # using for-loop to utilize regular expressions on list, all_sites is a list
    converted_strings = str(strings) # convert each element in list to string
    extracted_titles = re.findall(r'<h3 class="Cfb3RR">(.*?)</h3>', converted_strings) # using regular expressions to extract url pattern
    titles_list.append(extracted_titles) # append to empty list for urls

# Flattening the list
titles_list = [url for sublist in titles_list for url in sublist] # very bad solution for flatterning the list


In [18]:
### zipping lists together into df and exporting them as CSV file

df_overview = pd.DataFrame(zip(titles_list, url_list), columns=["Title", "URL"])


df_overview.to_csv('overview_of_borgerforslag.csv', index=False) # writing into csv file for export and sharing

In [19]:
len(df_overview)

1783

In [20]:
# Log for html extraction

import os

# Define the log function to gather the log information
def log(response,logfile,output_path=os.getcwd()):
    # Open or create the csv file
    if os.path.isfile(logfile): #If the log file exists, open it and allow for changes     
        log = open(logfile,'a')
    else: #If the log file does not exist, create it and make headers for the log variables
        log = open(logfile,'w')
        header = ['timestamp','status_code','length','output_file']
        log.write(';'.join(header) + "\n") #Make the headers and jump to new line
        
    # Gather log information
    status_code = response.status_code #Status code from the request result
    timestamp = time.strftime('%Y-%m-%d %H:%M:%S', time.localtime(time.time())) #Local time
    length = len(response.text) #Length of the HTML-string
    
    # Open the log file and append the gathered log information
    with open(logfile,'a') as log:
        log.write(f'{timestamp};{status_code};{length};{output_path}' + "\n") #Append the information and jump to new line

        

In [36]:
storing_raw_html = [] # creating empty list for storing html code of borgerforslag
for url in tqdm.tqdm(df_overview['URL']):
    try:
        html_borger = requests.get(url, headers={'Navn':'Sofus Møller', 'Email': 'qvc730@samf.ku.dk'})
    except Exception as e:
        print(url)
        print(e)
        with open('storing_raw_html', 'a') as l:
            json.dump(storing_raw_html, l)
        continue
    storing_raw_html.append(html_borger)
    log(html_borger, 'logfile_borgerforslag.csv')
    time.sleep(random.uniform(2,6))

# wait to load until a log is included (look at session #?)
# not all right now? 

  9%|▉         | 160/1783 [21:09<3:34:38,  7.93s/it]


KeyboardInterrupt: 

In [38]:
storing_raw_html.to_csv('raw_html.csv', index=False) # writing into csv file for export and sharing


AttributeError: 'list' object has no attribute 'to_csv'

In [50]:
# defining the file path to raw_html
raw_data = Path.cwd() / "raw_data" / "raw_html.csv"

# making into df
df_storing_raw_html = pd.DataFrame(storing_raw_html)

# Ensure the directory exists
raw_data.parent.mkdir(parents=True, exist_ok=True)

# Check if the file already exists
if not raw_data.exists():
    # If the file doesn't exist, save the DataFrame to CSV
    df_storing_raw_html.to_csv(raw_data, index=False)
    print(f"Data written to '{raw_data}' successfully.")
else:
    print(f"File '{raw_data}' already exists. Operation aborted.")

File '/Users/sofusgm/Documents/GitHub/University/My_ISDS/Borgerforslag/raw_data/raw_html.csv' already exists. Operation aborted.


In [57]:
raw_data_v2 = Path.cwd() / "raw_data" / "raw_html_v2.csv"
if not raw_data_v2.exists():
    # If the file doesn't exist, open it and write the list to it
    with raw_data_v2.open(mode='w', newline='') as file:
        writer = csv.writer(file)
        for item in storing_raw_html:
            writer.writerow([item])  # Writing each item in its own row
